In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib:", matplotlib.__version__)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True,
                     "grid.alpha": 0.2, "figure.dpi": 110})

# Week 2 · Task 1 — Data Preprocessing and Cleaning
**Student surname:** Gumar  
**Seed (last four digits of student ID): 3242**  
MOP 3231 · Machine Learning (Python)

This dataset contains synthetic apartment listings. Its prices are not evidence about the real Almaty market.
The goal is to keep one most recent posting per listing, document the cleaning rules, and prepare features without fitting imputers or scalers on the test set.

## Step 2 — Load and inspect

In [ ]:
YOUR_ID = 3242
raw = pd.read_csv("data/almaty_apartments_raw.csv")
ids = raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=YOUR_ID)
df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)
print("Raw shape:", raw.shape)
print("Sample shape:", df.shape)
print("Selected unique listings:", len(ids))
print("Types:"); print(df.dtypes)
print("Missing values:"); print(df.isna().sum())
display(df.describe(include="all").T)
for col in ["listed_date", "rooms", "area_m2", "price_kzt"]:
    print(col, "examples:", df[col].sample(10, random_state=YOUR_ID).tolist())
print("Studios:", df.loc[df.rooms.eq("studio"), "rooms"].head().tolist())
print("Areas with units or commas:", df.loc[df.area_m2.str.contains(r"m²|,", regex=True), "area_m2"].head(8).tolist())
print("Prices with formatting:", df.loc[df.price_kzt.str.contains(r"₸|,|\s", regex=True), "price_kzt"].head(8).tolist())
exact_count = int(df.duplicated().sum())
distinct = df.drop_duplicates()
repost_count = int(distinct["listing_id"].duplicated().sum())
print("Exact duplicate rows:", exact_count)
print("Additional nonidentical postings of the same listing:", repost_count)
pair_ids = distinct.loc[distinct["listing_id"].duplicated(keep=False), "listing_id"].drop_duplicates().head(2)
display(distinct[distinct.listing_id.isin(pair_ids)].sort_values(["listing_id", "listed_date"]))
for listing in pair_ids:
    pair = distinct[distinct.listing_id.eq(listing)]
    print(listing, "differing columns:", pair.columns[pair.nunique(dropna=False).gt(1)].tolist())

**Types:** `listed_date` is text because `read_csv` does not automatically parse dates. `rooms` includes the word `studio`. `area_m2` includes decimal commas and the unit `m²`. `price_kzt` includes thousands separators, spaces and the tenge symbol. `listing_id`, `district` and `building_type` are correctly stored as text before category cleaning.

**Grain:** one cleaned row represents the most recent dated posting of one unique `listing_id`. The displayed reposts differ in posting date and asking price; file order does not indicate which posting is newer.

### Validation specification
These are checks for this synthetic completed-apartment dataset, not universal property-market limits. Missing predictor values are allowed before imputation; unusually large but possible values must be inspected rather than automatically removed.

| Column | Expected type | Valid values / rule |
|---|---|---|
| listing_id | string | Nonmissing `ALM-` followed by digits; unique after repost resolution |
| listed_date | datetime | Valid date, no later than 2026-09-27 |
| district | category/string | Alatau, Almaly, Auezov, Bostandyk, Medeu, Nauryzbay, Turksib, Zhetysu |
| rooms | integer | 1–20; studio is defined as 1 room |
| area_m2 | float | Positive; review values above 200 m² together with rooms and district price per m² |
| floor | numeric integer or NaN | 1 ≤ floor ≤ total_floors |
| total_floors | integer | 1–100 storeys |
| year_built | numeric integer or NaN | 1800–2026 and no later than the listing year |
| building_type | category/string | panel, brick, monolith |
| ceiling_height_m | float or NaN | 2–6 metres |
| price_kzt | numeric | Positive asking price in KZT |

## Step 3 — Clean and transform

In [ ]:
cleaning_log = []
def log(step, operation, before, after, cells=0):
    cleaning_log.append({"Step": step, "Operation": operation,
                         "Rows before": int(before), "Rows after": int(after),
                         "Cells changed": int(cells)})

n = len(df)
df = df.drop_duplicates().copy()
log("3a", "Remove exact duplicate rows", n, len(df))
old = df["listed_date"].copy()
df["listed_date"] = pd.to_datetime(old, errors="coerce")
new_missing = int((old.notna() & df["listed_date"].isna()).sum())
print("New NaNs from date parsing:", new_missing)
assert new_missing == 0
log("3a", "Parse listing dates", len(df), len(df), old.notna().sum())
# The observed data has no ambiguous same-date reposts after exact deduplication.
assert not df.duplicated(["listing_id", "listed_date"]).any()
n = len(df)
df = df.sort_values(["listed_date", "listing_id"]).drop_duplicates("listing_id", keep="last").reset_index(drop=True)
log("3a", "Keep latest dated posting per listing", n, len(df))
print("Reposts resolved:", n-len(df))
print("Remaining duplicate listing IDs:", df.listing_id.duplicated().sum())

Exact copies repeat the same information, so they are removed. For reposts, I keep the posting with the latest parsed date because it gives the most recent asking price. The date check above ensures there is no unresolved same-date conflict.

In [ ]:
print("Original district spellings:")
print(df["district"].value_counts().to_string())
canonical = ["Alatau", "Almaly", "Auezov", "Bostandyk", "Medeu", "Nauryzbay", "Turksib", "Zhetysu"]
prefix_map = {
    "alata": "Alatau", "almal": "Almaly", "auezo": "Auezov", "bosta": "Bostandyk",
    "medeu": "Medeu", "naury": "Nauryzbay", "turks": "Turksib", "zhety": "Zhetysu",
    "алата": "Alatau", "алмал": "Almaly", "ауэзо": "Auezov", "боста": "Bostandyk",
    "медеу": "Medeu", "науры": "Nauryzbay", "туркс": "Turksib", "жетыс": "Zhetysu"
}
old = df["district"].copy()
df["district"] = old.str.strip().str.casefold().str[:5].map(prefix_map)
print("Unmapped spellings:", old[df["district"].isna()].unique().tolist())
assert df["district"].isna().sum() == 0
assert set(df["district"].unique()) == set(canonical)
log("3b", "Normalize district spelling/case/whitespace", len(df), len(df), old.ne(df.district).sum())
print("Canonical district count:", df.district.nunique())
print("Missing districts:", df.district.isna().sum())

parsed = {
    "rooms": df["rooms"].str.strip().str.casefold().replace({"studio": "1"}),
    "area_m2": df["area_m2"].str.replace("m²", "", regex=False).str.strip().str.replace(",", ".", regex=False),
    "price_kzt": df["price_kzt"].str.replace(r"[₸,\s]", "", regex=True)
}
for col, values in parsed.items():
    old = df[col].copy()
    df[col] = pd.to_numeric(values, errors="coerce")
    new_missing = int((old.notna() & df[col].isna()).sum())
    print(col, "new NaNs from parsing:", new_missing)
    assert new_missing == 0, f"Unparsed values in {col}"
    log("3c", f"Parse {col} to numeric", len(df), len(df), old.notna().sum())
df["rooms"] = df["rooms"].astype(int)
print(df.dtypes)
display(pd.DataFrame(cleaning_log))

I treat a studio as 1 room. A comma in area is a decimal separator, so `65,5` becomes `65.5`; commas in prices are thousands separators and are removed. Unit symbols and price whitespace are removed without deleting decimal points or signs. The parsing checks report unexpected new missing values rather than hiding them.

## Step 4 — Sentinels, impossible values and outliers

In [ ]:
display(df.describe().T)
for col in ["floor", "year_built", "ceiling_height_m"]:
    print(col, "smallest values and frequencies:")
    print(df[col].value_counts(dropna=False).sort_index().head(6))
sentinels = {"floor": -1, "year_built": 0, "ceiling_height_m": 0}
for col, sentinel in sentinels.items():
    mask = df[col].eq(sentinel)
    count = int(mask.sum())
    df.loc[mask, col] = np.nan
    log("4", f"Replace {col} sentinel {sentinel} with NaN", len(df), len(df), count)
    print(col, "sentinels replaced:", count)

impossible_floor = df["floor"] > df["total_floors"]
print("Impossible floor > total_floors:", impossible_floor.sum())
display(df.loc[impossible_floor, ["listing_id", "floor", "total_floors"]])
count_impossible = int(impossible_floor.sum())
df.loc[impossible_floor, "floor"] = np.nan
log("4", "Set contradictory floor to NaN", len(df), len(df), count_impossible)

checks = {
    "bad listing_id": ~df.listing_id.str.fullmatch(r"ALM-\d+", na=False),
    "duplicate listing_id": df.listing_id.duplicated(),
    "bad listed_date": df.listed_date.isna() | df.listed_date.gt(pd.Timestamp("2026-09-27")),
    "bad district": ~df.district.isin(canonical),
    "bad rooms": ~df.rooms.between(1, 20) | df.rooms.mod(1).ne(0),
    "bad area": df.area_m2.isna() | df.area_m2.le(0),
    "bad floor": df.floor.notna() & (~df.floor.ge(1) | df.floor.gt(df.total_floors) | df.floor.mod(1).ne(0)),
    "bad total_floors": ~df.total_floors.between(1, 100) | df.total_floors.mod(1).ne(0),
    "bad year_built": df.year_built.notna() & (~df.year_built.between(1800, 2026) | df.year_built.gt(df.listed_date.dt.year) | df.year_built.mod(1).ne(0)),
    "bad building_type": ~df.building_type.isin(["panel", "brick", "monolith"]),
    "bad ceiling": df.ceiling_height_m.notna() & ~df.ceiling_height_m.between(2, 6),
    "bad price": df.price_kzt.isna() | df.price_kzt.le(0)
}
print("Remaining validation violations:")
print(pd.Series({name: int(mask.sum()) for name, mask in checks.items()}))
assert not any(mask.any() for mask in checks.values())

The sentinel values are placeholders, not real measurements. For a floor above the number of storeys, either field could be wrong. I set the apartment floor to missing and retain the otherwise valid row and building height; this is an explicit working assumption, not a claim that the true floor is known.

In [ ]:
df["price_per_m2"] = df["price_kzt"] / df["area_m2"]
df["area_before_fix"] = df["area_m2"]
def fences(values, groups=None):
    grouped = values if groups is None else values.groupby(groups)
    q1, q3 = grouped.quantile(0.25), grouped.quantile(0.75)
    if groups is not None:
        q1, q3 = groups.map(q1), groups.map(q3)
    iqr = q3-q1
    return q1-1.5*iqr, q3+1.5*iqr

global_low, global_high = fences(df["price_per_m2"])
global_flag = (df.price_per_m2 < global_low) | (df.price_per_m2 > global_high)
low, high = fences(df["price_per_m2"], df["district"])
group_flag = (df.price_per_m2 < low) | (df.price_per_m2 > high)
print(f"Global fences: {global_low:,.0f} to {global_high:,.0f} KZT/m²")
print("Globally flagged rows:", int(global_flag.sum()))
print("District-level flagged rows:", int(group_flag.sum()))
plt.hist(df.price_per_m2 / 1000, bins=40, color="#4267a3", edgecolor="white")
plt.xlabel("Price per m² (thousand KZT/m²), before correction")
plt.ylabel("Number of listings")
plt.title("Diagnostic histogram: mixed district price levels")
plt.tight_layout(); plt.show()
display(df.loc[group_flag, ["listing_id", "district", "rooms", "area_m2", "price_kzt", "price_per_m2"]].sort_values("price_per_m2"))

Global IQR fences combine districts with different price levels, making the interval too wide to detect some locally abnormal prices per m². District-level fences compare more similar listings. I use them as inspection flags; they do not automatically justify deleting a row.

In [ ]:
# A proposed factor-of-ten area correction must pass three independent plausibility checks.
candidate_area = df["area_m2"] / 10
candidate_ppm = df["price_kzt"] / candidate_area
fix_area = (group_flag & df.price_per_m2.lt(low)
            & df.area_m2.gt(200) & (df.area_m2 / df.rooms).gt(100)
            & (candidate_area / df.rooms).between(10, 60)
            & candidate_ppm.ge(low) & candidate_ppm.le(high))
audit = df.loc[fix_area, ["listing_id", "district", "rooms", "area_m2", "price_per_m2"]].copy()
audit["area_after_m2"] = candidate_area[fix_area]
audit["price_per_m2_after"] = candidate_ppm[fix_area]
audit["district_low"] = low[fix_area]
audit["district_high"] = high[fix_area]
print("Area corrections:", int(fix_area.sum()))
display(audit)
assert audit.price_per_m2_after.between(audit.district_low, audit.district_high).all()
kept_flagged = df.loc[group_flag & ~fix_area, ["listing_id", "district", "rooms", "area_m2", "price_kzt", "price_per_m2"]].copy()
print("Flagged rows retained without correction:", len(kept_flagged))
display(kept_flagged)
df.loc[fix_area, "area_m2"] = candidate_area[fix_area]
log("4", "Correct supported factor-of-ten area errors", len(df), len(df), fix_area.sum())
old_ppm = df["price_per_m2"].copy()
df["price_per_m2"] = df["price_kzt"] / df["area_m2"]
log("4", "Recompute price_per_m2 after area corrections", len(df), len(df), old_ppm.ne(df.price_per_m2).sum())
log("4", "Keep plausible statistical outliers; drop none", len(df), len(df), 0)
df.to_csv("data/almaty_apartments_clean.csv", index=False)
print("Saved clean data before imputation:", df.shape)
print("Remaining missing values:"); print(df.isna().sum())

**Correct:** the displayed huge areas occur in small room-count apartments and produce district prices per m² about one order of magnitude too low. Dividing area by 10 gives reasonable area per room and puts every corrected price per m² inside its original district fences, as the audit table checks. This supports a decimal-place error; it is an inference from the data, not externally confirmed ground truth.

**Keep:** the remaining flagged rows have plausible areas, room counts and positive prices. A locally unusual price alone is not sufficient evidence of an error, so I keep them unchanged. **Drop:** no row is dropped merely because it is an outlier.

`area_before_fix` is retained only for the requested scaler demonstration. `price_per_m2` uses the target and is never a model feature. The CSV above contains missing values and is saved before any imputation. The full-sample outlier audit follows Step 4 of the assignment; because it uses prices and sample-derived fences, the later R² is a lab sanity check rather than a completely untouched production holdout estimate.